# Milestone 2: Data Quality dan Profiling

**Dataset:** Daily Rainfall Observations from 460 Stations and Flood Event Records across West Java, Indonesia (2020-2025)  
**Sumber:** BMKG + BNPB/BPBD (Mendeley Data DOI: 10.17632/d7j3s64vvr.3)  
**Tujuan:** Melakukan data cleaning, profiling dengan Polars + DuckDB, transformasi wide->long, penanganan missing values, outlier, deduplikasi, validasi, dan menyimpan hasil bersih ke `prosessed_data/`

## 1. Import Library & Setup Path

In [1]:
from pathlib import Path
import duckdb
import polars as pl

ROOT = Path(r"C:\project_1\data analysis\tugas 1")
DATASET_DIR = ROOT / "dataset"
RAW_XLSX = DATASET_DIR / "01_Data-Ch-Jabar-2020-2025.xlsx"
META_XLSX = DATASET_DIR / "02_West_Java_Rainfall_Station_Metadata_Lengkap.xlsx"
OUT_DIR = ROOT / "prosessed_data"
OUT_PARQUET = OUT_DIR / "west_java_rainfall_long_clean.parquet"
OUT_CSV_SAMPLE = OUT_DIR / "west_java_rainfall_long_clean_sample_10000.csv"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT: {ROOT}")
print(f"Output dir: {OUT_DIR}")

ROOT: C:\project_1\data analysis\tugas 1
Output dir: C:\project_1\data analysis\tugas 1\prosessed_data


## 2. Load Data Wide Format

In [2]:
print("[1/9] Loading wide matrix from Excel...")
df_wide = pl.read_excel(RAW_XLSX, sheet_name="DATA")
print(f"Wide shape: {df_wide.shape[0]:,} rows x {df_wide.shape[1]:,} cols")
df_wide.head()

[1/9] Loading wide matrix from Excel...


Could not determine dtype for column 35, falling back to string


Could not determine dtype for column 112, falling back to string


Could not determine dtype for column 205, falling back to string


Could not determine dtype for column 217, falling back to string


...

Could not determine dtype for column 426, falling back to string


Could not determine dtype for column 453, falling back to string


Could not determine dtype for column 460, falling back to string


Wide shape: 2,192 rows x 461 cols


Date,Ciriung,BP3K Jonggol,Dayeuh/Sukanagara,Klapanunggal,Situ Tungilis,Cariu,Sinar Rasa_Tanjung rasa,Jampang,Curug Serpong,Situ Kemang,Bojong Gede,Cis Empang,Citayam,Perkeb. Cianten,Pamijahan,PLTA Kracak_Power House,Bp3K Leuwiliang,Dam Cianten,Cihideung,Cibodas,Cikluwung,Citeureup Pamijahan,Perkeb. Jasinga,Cigelung,Parung Panjang,Nanggung,Afd Bolang,Perkeb. Cikasungka,Empl. Toge,Ciawi,Naringgul,Cibeureum Bogor,Perk Gunung Mas,Gadog,Pondok Gedeh,Cibalagung Bogor,…,Padalarang,Cililin,Sindangkerta,Perk. Montaya,Kondang Jajar_Perhubpindo_Cijulang_Nusawiru,Afd Cimerak,Empang,Katulampa,Kb. Percob. Cimanggu,Kebun Raya Bogor,Darmaga,Ciaul,Cadin_PSDA Sukabumi,Situ mekar,Rawa Dukuh,Kranji,Bendung Bekasi,Cikeas,Lab.Perikanan Depok,Beji Depok,Cigede_Sindang galih,Talagasari_Kawalu,Afd. Mandalareh,Lanud Atang Sanjaya,Citeko,Pacet Cianjur,Lanud Suryadarma_Kalijati Subang,Lanud_Wiriadinata,Sta. Met Jatiwangi,Buah Dua,Bandung_Saguling ( puslit air ),PLTA Bengkok,Lengkong/ Telaga Bodas,Lanud Husein,Sta. Geofisika Bandung,SMPK Sukapura,Cimahi
date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,…,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,str
2020-01-01,45.0,0.0,55.0,55.0,63.0,127.0,82.0,32.0,36.0,75.0,60.0,65.0,243.0,74.5,85.0,NaN,70.0,NaN,60.0,34.0,NaN,85.0,61.0,30.0,38.0,150.0,35.0,275.0,135.0,120.0,40.0,83.6,66.0,110.0,"""NaN""",110.0,…,85.7,0.0,"""NaN""",24.0,4.0,14.0,65.0,49.0,102.0,141.0,44.4,59.0,65.0,56.0,155.0,120.0,150.0,246.0,125.0,142.0,5.0,4.0,19.0,43.4,60.0,39.0,11.0,8.7,50.5,"""NaN""",1.5,0.0,14.5,0.2,13.4,44.0,"""NaN"""
2020-01-02,106.0,7.0,40.0,36.0,38.0,20.0,3.0,0.0,0.0,18.0,0.0,10.0,16.0,73.0,6.0,NaN,15.0,NaN,30.0,30.0,NaN,6.0,14.0,15.0,9.5,100.0,50.0,24.0,9.5,60.5,69.0,85.2,58.5,34.0,"""NaN""",0.0,…,2.5,8.0,"""NaN""",8.5,10.0,16.0,47.0,74.0,53.5,0.2,34.3,16.0,22.0,15.0,55.0,99.0,54.0,36.5,107.0,54.0,2.0,6.0,0.0,0.0,73.0,29.0,1.0,26.3,6.3,"""NaN""",0.0,0.0,1.0,0.0,2.2,0.0,"""NaN"""
2020-01-03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,1.8,0.0,0.0,NaN,1.0,NaN,0.0,0.0,NaN,0.0,25.0,30.0,4.0,15.0,13.0,1.0,8.5,0.0,7.0,8.2,9.5,0.0,"""NaN""",27.0,…,0.0,0.0,"""NaN""",8.0,2.5,18.0,0.0,0.0,0.0,0.2,0.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.5,0.0,14.0,1.5,0.0,0.0,6.8,0.0,0.0,0.0,0.0,"""NaN""",0.0,0.0,0.0,0.0,0.0,0.0,"""NaN"""
2020-01-04,0.0,0.0,38.0,45.0,26.0,0.0,0.0,14.0,9.0,0.0,0.0,0.0,0.1,5.5,0.0,NaN,18.0,NaN,20.0,0.0,NaN,0.0,3.0,5.0,17.0,0.0,2.0,1.0,4.0,0.0,1.0,1.1,0.5,2.0,"""NaN""",0.4,…,0.0,0.0,"""NaN""",0.0,2.0,0.0,0.0,0.0,3.5,17.6,0.3,0.0,0.0,0.0,0.0,2.0,2.0,2.5,0.0,0.0,2.0,0.0,0.0,12.9,0.8,0.0,6.0,9.8,0.0,"""NaN""",0.0,0.0,0.0,0.0,0.0,0.0,"""NaN"""
2020-01-05,26.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,0.0,9.0,12.0,0.0,0.0,7.0,10.0,NaN,6.0,NaN,0.0,29.0,NaN,10.0,0.0,15.0,34.5,0.0,0.0,4.5,12.5,0.0,13.0,5.3,15.5,2.5,"""NaN""",0.7,…,0.0,0.0,"""NaN""",1.0,3.0,0.0,5.0,5.0,1.0,4.8,3.3,12.0,20.0,5.0,7.0,5.0,0.0,0.0,0.0,0.0,4.0,18.0,9.0,0.0,5.8,0.0,0.0,16.0,0.0,"""NaN""",0.0,0.0,0.0,0.0,0.0,3.0,"""NaN"""


## 3. Identifikasi Kolom Tanggal & Stasiun

In [3]:
date_col = df_wide.columns[0]
station_cols = [c for c in df_wide.columns if c != date_col]
print(f"Date col: {date_col}")
print(f"Jumlah stasiun: {len(station_cols)}")
print(f"Stasiun contoh: {station_cols[:5]}")

Date col: Date
Jumlah stasiun: 460
Stasiun contoh: ['Ciriung', 'BP3K Jonggol', 'Dayeuh/Sukanagara', 'Klapanunggal', 'Situ Tungilis']


## 4. Transformasi Wide -> Long Format

Matriks 2,192 hari x 460 stasiun akan di-unpivot menjadi bentuk long: `(tanggal, station_id, rainfall_mm)`. Hasilnya akan menghasilkan ~1.008.320 baris sehingga memenuhi syarat **> 1.000.000 baris**.

In [4]:
print("[2/9] Melting wide -> long format...")
df_long = df_wide.unpivot(
    index=[date_col],
    on=station_cols,
    variable_name="station_id",
    value_name="rainfall_mm"
).rename({date_col: "tanggal"})

df_long = df_long.with_columns([
    pl.col("tanggal").cast(pl.Date),
    pl.col("rainfall_mm").cast(pl.Utf8).str.strip_chars().alias("rainfall_mm_str")
])

# Handle trace/symbolic values from BMKG (R, TR, T, -, empty)
df_long = df_long.with_columns([
    pl.when(
        pl.col("rainfall_mm_str").is_null() |
        (pl.col("rainfall_mm_str") == "") |
        pl.col("rainfall_mm_str").str.to_lowercase().is_in(["nan", "none", "-", "na", "#n/a"])
    )
    .then(None)
    .when(pl.col("rainfall_mm_str").str.to_lowercase().is_in(["r", "tr", "t", "trace", "h"])
    )
    .then(0.0)
    .otherwise(pl.col("rainfall_mm_str").str.replace_all(r"[^0-9\.\-]", ""))
    .alias("rainfall_mm_clean_str")
]).with_columns([
    pl.when(pl.col("rainfall_mm_clean_str") == "")
    .then(None)
    .otherwise(pl.col("rainfall_mm_clean_str").cast(pl.Float64, strict=False))
    .alias("rainfall_mm")
]).drop(["rainfall_mm_str", "rainfall_mm_clean_str"])

print(f"Long shape (raw melt): {df_long.shape[0]:,} rows x {df_long.shape[1]:,} cols")
print(f"Expected: {len(df_wide):,} * {len(station_cols)} = {len(df_wide)*len(station_cols):,} rows")
df_long.head()


[2/9] Melting wide -> long format...


Long shape (raw melt): 1,008,320 rows x 3 cols
Expected: 2,192 * 460 = 1,008,320 rows


tanggal,station_id,rainfall_mm
date,str,f64
2020-01-01,"""Ciriung""",45.0
2020-01-02,"""Ciriung""",106.0
2020-01-03,"""Ciriung""",0.0
2020-01-04,"""Ciriung""",0.0
2020-01-05,"""Ciriung""",26.0


## 5. Join dengan Metadata Stasiun

In [5]:
print("[3/9] Loading station metadata & joining...")
try:
    df_meta = pl.read_excel(META_XLSX, sheet_name="metadata")
    df_meta = df_meta.rename({c: c.strip().lower().replace(" ", "_").replace("/", "_") for c in df_meta.columns})
    if "station_id" not in df_meta.columns and "id_stasiun" in df_meta.columns:
        df_meta = df_meta.rename({"id_stasiun": "station_id"})
    elif "station_id" not in df_meta.columns and "kode_stasiun" in df_meta.columns:
        df_meta = df_meta.rename({"kode_stasiun": "station_id"})
    elif "station_id" not in df_meta.columns and "nama_stasiun" in df_meta.columns:
        # try to match by name if needed - but better use existing
        pass
    print(f"Metadata shape: {df_meta.shape}")
    df_long = df_long.join(df_meta, on="station_id", how="left")
    print(f"After join shape: {df_long.shape}")
    print("Metadata joined successfully")
except Exception as e:
    print(f"[WARN] Gagal join metadata: {e}")
    print("Lanjut tanpa metadata")

df_long.head()

[3/9] Loading station metadata & joining...
Metadata shape: (460, 8)
After join shape: (1008320, 10)
Metadata joined successfully


tanggal,station_id,rainfall_mm,district,sub_district,rainfall_station,latitude,longitude,elev_(_m_),missing_data_(%)
date,str,f64,str,str,str,f64,f64,i64,f64
2020-01-01,"""Ciriung""",45.0,null,null,null,null,null,null,null
2020-01-02,"""Ciriung""",106.0,null,null,null,null,null,null,null
2020-01-03,"""Ciriung""",0.0,null,null,null,null,null,null,null
2020-01-04,"""Ciriung""",0.0,null,null,null,null,null,null,null
2020-01-05,"""Ciriung""",26.0,null,null,null,null,null,null,null


## 6. Profiling dengan DuckDB SUMMARIZE

In [6]:
print("[4/9] Profiling with DuckDB SUMMARIZE...")
con = duckdb.connect()
con.register("df_long", df_long.to_pandas())
try:
    summ = con.execute("SUMMARIZE SELECT * FROM df_long").df()
    display(summ[["column_name","column_type","min","max","approx_unique","avg","count","null_percentage"]].head(20))
except Exception as e:
    print(f"[WARN] SUMMARIZE error: {e}")
finally:
    con.close()

[4/9] Profiling with DuckDB SUMMARIZE...


,column_name,column_type,min,max,approx_unique,avg,count,null_percentage
0,tanggal,TIMESTAMP_MS,2020-01-01 00:00:00,2025-12-31 00:00:00,2231,2022-12-31 12:00:00,1008320,0.00
1,station_id,VARCHAR,Adf. Ciaren_Batu Lawang,Werasari,460,NaN,1008320,0.00
2,rainfall_mm,DOUBLE,0.0,456.0,1280,7.627714233298821,1008320,2.15
3,district,VARCHAR,NaN,NaN,0,NaN,1008320,100.00
4,sub_district,VARCHAR,NaN,NaN,0,NaN,1008320,100.00
5,rainfall_station,VARCHAR,NaN,NaN,0,NaN,1008320,100.00
6,latitude,DOUBLE,NaN,NaN,0,NaN,1008320,100.00
7,longitude,DOUBLE,NaN,NaN,0,NaN,1008320,100.00
8,elev_(_m_),DOUBLE,NaN,NaN,0,NaN,1008320,100.00
9,missing_data_(%),DOUBLE,NaN,NaN,0,NaN,1008320,100.00


## 7. Deteksi Missing Values

In [7]:
print("[5/9] Detecting missing values...")
jumlah_baris = df_long.height
null_summary = (
    df_long.null_count()
    .transpose(include_header=True, header_name='kolom', column_names=['jumlah_null'])
    .with_columns(
        (pl.col('jumlah_null') / jumlah_baris * 100).round(2).alias('persen_null')
    )
    .filter(pl.col('jumlah_null') > 0)
    .sort('jumlah_null', descending=True)
)
display(null_summary)
print(f"\nTotal null: {df_long.null_count().sum_horizontal().item():,}")
print("[INFO] rainfall_mm NaN dipertahankan sesuai prinsip raw observational data (README).")

[5/9] Detecting missing values...


kolom,jumlah_null,persen_null
str,u32,f64
"""district""",1008320,100.0
"""sub_district""",1008320,100.0
"""rainfall_station""",1008320,100.0
"""latitude""",1008320,100.0
"""longitude""",1008320,100.0
"""elev_(_m_)""",1008320,100.0
"""missing_data_(%)""",1008320,100.0
"""rainfall_mm""",21697,2.15



Total null: 7,079,937
[INFO] rainfall_mm NaN dipertahankan sesuai prinsip raw observational data (README).


## 8. Outlier Detection & Winsorization (IQR)

In [8]:
print("[6/9] Outlier handling with IQR + winsorization...")
q = df_long.filter(pl.col("rainfall_mm").is_not_null() & (pl.col("rainfall_mm") >= 0)) \
          .select([
              pl.col("rainfall_mm").quantile(0.25).alias("q1"),
              pl.col("rainfall_mm").quantile(0.75).alias("q3")
          ]).row(0)
q1, q3 = q[0], q[1]
iqr = q3 - q1
lower = q1 - 1.5*iqr
upper = q3 + 1.5*iqr
print(f"Q1={q1:.2f} | Q3={q3:.2f} | IQR={iqr:.2f}")
print(f"IQR bounds: [{lower:.2f}, {upper:.2f}]")

df_out = df_long.with_columns([
    pl.when(pl.col("rainfall_mm").is_between(lower, upper, closed="both"))
      .then(pl.col("rainfall_mm"))
      .when(pl.col("rainfall_mm") > upper).then(upper)
      .when(pl.col("rainfall_mm") < lower).then(lower)
      .otherwise(pl.col("rainfall_mm"))
      .alias("rainfall_mm_clipped")
])
print("[INFO] rainfall_mm_clipped dibuat (winsorized/clipped)")
df_out.head()

[6/9] Outlier handling with IQR + winsorization...
Q1=0.00 | Q3=7.50 | IQR=7.50
IQR bounds: [-11.25, 18.75]
[INFO] rainfall_mm_clipped dibuat (winsorized/clipped)


tanggal,station_id,rainfall_mm,district,sub_district,rainfall_station,latitude,longitude,elev_(_m_),missing_data_(%),rainfall_mm_clipped
date,str,f64,str,str,str,f64,f64,i64,f64,f64
2020-01-01,"""Ciriung""",45.0,null,null,null,null,null,null,null,18.75
2020-01-02,"""Ciriung""",106.0,null,null,null,null,null,null,null,18.75
2020-01-03,"""Ciriung""",0.0,null,null,null,null,null,null,null,0.0
2020-01-04,"""Ciriung""",0.0,null,null,null,null,null,null,null,0.0
2020-01-05,"""Ciriung""",26.0,null,null,null,null,null,null,null,18.75


## 9. Deduplikasi & Validasi

In [9]:
print("[7/9] Deduplication & validation...")
key = ["tanggal", "station_id"]
before_dedup = df_out.height
df_dedup = df_out.unique(subset=key, keep="first")
after_dedup = df_dedup.height
print(f"Rows before dedup by {key}: {before_dedup:,}")
print(f"Rows after dedup: {after_dedup:,} | Removed: {before_dedup-after_dedup:,}")

valid = df_dedup.select([
    pl.len().alias("total_rows"),
    pl.col("rainfall_mm").is_null().sum().alias("null_rainfall"),
    pl.col("rainfall_mm_clipped").is_null().sum().alias("null_rainfall_clipped"),
    ((pl.col("rainfall_mm") < 0) & pl.col("rainfall_mm").is_not_null()).sum().alias("neg_rainfall"),
    pl.col("tanggal").is_null().sum().alias("null_tanggal"),
    pl.col("station_id").is_null().sum().alias("null_station_id"),
])
display(valid)

[7/9] Deduplication & validation...
Rows before dedup by ['tanggal', 'station_id']: 1,008,320
Rows after dedup: 1,008,320 | Removed: 0


total_rows,null_rainfall,null_rainfall_clipped,neg_rainfall,null_tanggal,null_station_id
u32,u32,u32,u32,u32,u32
1008320,21697,21697,0,0,0


## 10. Simpan Data Bersih ke prosessed_data/

In [10]:
print("[8/9] Preparing data to save...")
# Prioritaskan kolom penting
priority_cols = ["tanggal", "station_id", "rainfall_mm", "rainfall_mm_clipped"]
other_cols = [c for c in df_dedup.columns if c not in priority_cols]
df_save = df_dedup.select(priority_cols + other_cols)

print(f"Final shape: {df_save.shape[0]:,} rows x {df_save.shape[1]:,} cols")
df_save.head()

[8/9] Preparing data to save...
Final shape: 1,008,320 rows x 11 cols


tanggal,station_id,rainfall_mm,rainfall_mm_clipped,district,sub_district,rainfall_station,latitude,longitude,elev_(_m_),missing_data_(%)
date,str,f64,f64,str,str,str,f64,f64,i64,f64
2020-12-28,"""Rawamerta""",0.0,0.0,null,null,null,null,null,null,null
2025-10-23,"""Cayur_Cikatomas""",42.0,18.75,null,null,null,null,null,null,null
2021-09-11,"""Cinangling""",0.0,0.0,null,null,null,null,null,null,null
2021-12-14,"""Perk. Kahuripan""",1.0,1.0,null,null,null,null,null,null,null
2023-10-10,"""Bankir""",0.0,0.0,null,null,null,null,null,null,null


In [11]:
print("[9/9] Saving processed data...")
df_save.write_parquet(OUT_PARQUET)
print(f"SAVED: {OUT_PARQUET}")
df_save.head(10000).write_csv(OUT_CSV_SAMPLE)
print(f"SAMPLE: {OUT_CSV_SAMPLE}")
print(f"\nFile size: {OUT_PARQUET.stat().st_size / (1024**2):.2f} MB")
print("\n" + "="*60)
print("MILESTONE 2 SELESAI")
print("="*60)
print(f"Total rows: {df_save.shape[0]:,} (> 1,000,000: {'YES' if df_save.shape[0] > 1000000 else 'NO'})")
print("Next: Lanjut ke Milestone 3 (notebooks/03_eda_and_insights.ipynb)")

[9/9] Saving processed data...


SAVED: C:\project_1\data analysis\tugas 1\prosessed_data\west_java_rainfall_long_clean.parquet
SAMPLE: C:\project_1\data analysis\tugas 1\prosessed_data\west_java_rainfall_long_clean_sample_10000.csv

File size: 4.52 MB

MILESTONE 2 SELESAI
Total rows: 1,008,320 (> 1,000,000: YES)
Next: Lanjut ke Milestone 3 (notebooks/03_eda_and_insights.ipynb)


## Catatan Milestone 2

- **Dimensi data**: Wide (2192 x 461) -> Long (~1,008,320 baris). Memenuhi syarat > 1.000.000 baris.
- **Missing values**: `rainfall_mm` memiliki nilai NaN (missing observation) sesuai README. Dipertahankan apa adanya (raw observational integrity). Tidak di-imputasi sembarangan.
- **Outlier**: Deteksi menggunakan metode IQR. Penanganan dengan **winsorization/clipping** (`rainfall_mm_clipped`) ke batas [Q1-1.5*IQR, Q3+1.5*IQR].
- **Deduplikasi**: Deduplikasi berdasarkan kunci bisnis `tanggal + station_id` dengan `keep='first'`.
- **Validasi**: Dicek null, nilai negatif, null pada kunci utama.
- **Tools**: Polars untuk transformasi/cleaning, DuckDB untuk `SUMMARIZE`.
- **Output**: Tersimpan di `prosessed_data/west_java_rainfall_long_clean.parquet` siap digunakan untuk Milestone 3.